# Evaluate Detector

This notebook evaluates the trained YOLOv8 dice detector on the validation split.

Goals:
- load the trained model and validation data
- run raw predictions on the validation set
- sweep confidence thresholds
- compute precision, recall, F1-score, mean IoU, and approximate AP@50
- save evaluation results to CSV
- visualize the main evaluation curves

In [ ]:
import json
import csv

import numpy as np
import tensorflow as tf
import keras
import keras_cv
import matplotlib.pyplot as plt

from project_config import (
    VAL_JSON,
    MODELS_DIR,
    DETECTOR_TARGET_SIZE,
    BOUNDING_BOX_FORMAT,
)
from utils.detector_data import (
    validate_and_fix_sample,
    load_image,
)
from utils.detector_eval import (
    greedy_match,
    compute_ap_from_pr,
)

In [ ]:
MODEL_PATH = MODELS_DIR / "dice_detector_best_20260330_114353.keras"
TRAINING_CSV_PATH = MODELS_DIR / "dice_detector_training_20260330_114353.csv"

EVAL_IOU_THRESHOLD = 0.5
THRESHOLDS = np.arange(0.05, 0.96, 0.05)

assert VAL_JSON.exists(), f"Missing VAL_JSON: {VAL_JSON}"
assert MODEL_PATH.exists(), f"Missing MODEL_PATH: {MODEL_PATH}"
assert TRAINING_CSV_PATH.exists(), f"Missing TRAINING_CSV_PATH: {TRAINING_CSV_PATH}"

print("VAL_JSON:", VAL_JSON)
print("MODEL_PATH:", MODEL_PATH)
print("TRAINING_CSV_PATH:", TRAINING_CSV_PATH)

In [ ]:
with open(VAL_JSON, "r", encoding="utf-8") as f:
    val_data = json.load(f)

print("Validation samples:", len(val_data))

if not val_data:
    raise ValueError(f"Validation split is empty: {VAL_JSON}")

In [ ]:
resize_layer = keras_cv.layers.Resizing(
    DETECTOR_TARGET_SIZE,
    DETECTOR_TARGET_SIZE,
    bounding_box_format=BOUNDING_BOX_FORMAT,
    pad_to_aspect_ratio=True,
)

def apply_eval_preprocessing(inputs):
    inputs = resize_layer(inputs)
    inputs["images"] = inputs["images"] / 255.0
    return inputs

In [ ]:
def predict_sample_raw(model, sample):
    image_path, boxes, classes = validate_and_fix_sample(sample)
    image = load_image(tf.constant(image_path)).numpy().astype(np.float32)

    inputs = {
        "images": tf.convert_to_tensor(image[None, ...]),
        "bounding_boxes": {
            "boxes": tf.ragged.constant([boxes], dtype=tf.float32),
            "classes": tf.ragged.constant([classes], dtype=tf.float32),
        },
    }

    inputs = apply_eval_preprocessing(inputs)
    image_resized = inputs["images"][0].numpy()
    gt_boxes = inputs["bounding_boxes"]["boxes"][0].numpy()

    preds = model.predict(image_resized[None, ...], verbose=0)
    num_det = int(preds["num_detections"][0]) if "num_detections" in preds else len(preds["boxes"][0])

    pred_boxes = np.asarray(preds["boxes"][0][:num_det], dtype=np.float32).reshape(-1, 4)
    pred_scores = np.asarray(preds["confidence"][0][:num_det], dtype=np.float32).reshape(-1)

    order = np.argsort(-pred_scores)
    pred_boxes = pred_boxes[order]
    pred_scores = pred_scores[order]

    return {
        "file": image_path.split("/")[-1],
        "gt_boxes": gt_boxes,
        "pred_boxes": pred_boxes,
        "pred_scores": pred_scores,
        "image_resized": image_resized,
    }

In [ ]:
def evaluate_at_threshold(predictions, threshold=0.5, match_iou_threshold=0.5):
    total_gt = 0
    total_pred = 0
    total_tp = 0
    total_fp = 0
    total_fn = 0

    matched_ious_all = []
    tp_scores = []
    fp_scores = []
    per_image_rows = []

    for p in predictions:
        gt_boxes = p["gt_boxes"]

        keep = p["pred_scores"] >= threshold
        pred_boxes = p["pred_boxes"][keep]
        pred_scores = p["pred_scores"][keep]

        matched_gt, matched_pred, matches = greedy_match(
            gt_boxes=gt_boxes,
            pred_boxes=pred_boxes,
            pred_scores=pred_scores,
            match_iou_threshold=match_iou_threshold,
        )

        tp = len(matches)
        fn = len(gt_boxes) - tp
        fp = len(pred_boxes) - tp

        total_gt += len(gt_boxes)
        total_pred += len(pred_boxes)
        total_tp += tp
        total_fp += fp
        total_fn += fn

        matched_ious_all.extend([m[2] for m in matches])

        for pred_idx, score in enumerate(pred_scores):
            if pred_idx in matched_pred:
                tp_scores.append(float(score))
            else:
                fp_scores.append(float(score))

        precision_img = tp / max(tp + fp, 1)
        recall_img = tp / max(tp + fn, 1)
        f1_img = (2 * precision_img * recall_img) / max(precision_img + recall_img, 1e-8)
        mean_iou_img = float(np.mean([m[2] for m in matches])) if matches else 0.0

        per_image_rows.append({
            "file": p["file"],
            "gt": int(len(gt_boxes)),
            "pred": int(len(pred_boxes)),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "precision": float(precision_img),
            "recall": float(recall_img),
            "f1": float(f1_img),
            "mean_iou": float(mean_iou_img),
        })

    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)
    f1 = (2 * precision * recall) / max(precision + recall, 1e-8)
    mean_iou = float(np.mean(matched_ious_all)) if matched_ious_all else 0.0

    return {
        "threshold": float(threshold),
        "total_gt": int(total_gt),
        "total_pred": int(total_pred),
        "total_tp": int(total_tp),
        "total_fp": int(total_fp),
        "total_fn": int(total_fn),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "mean_iou": float(mean_iou),
        "tp_scores": tp_scores,
        "fp_scores": fp_scores,
        "per_image_rows": per_image_rows,
    }

In [ ]:
model = keras.models.load_model(MODEL_PATH)
model.prediction_decoder = keras_cv.layers.NonMaxSuppression(
    bounding_box_format=BOUNDING_BOX_FORMAT,
    from_logits=False,
    confidence_threshold=0.01,
    iou_threshold=0.5,
)

print("Loaded model:", MODEL_PATH)

In [ ]:
raw_predictions = [predict_sample_raw(model, sample) for sample in val_data]
print("Collected raw predictions:", len(raw_predictions))

In [ ]:
sweep_results = []
for thr in THRESHOLDS:
    res = evaluate_at_threshold(
        raw_predictions,
        threshold=thr,
        match_iou_threshold=EVAL_IOU_THRESHOLD,
    )
    sweep_results.append(res)

best_result = max(sweep_results, key=lambda x: x["f1"])
best_thr = best_result["threshold"]
ap50 = compute_ap_from_pr(
    recalls=[r["recall"] for r in sweep_results],
    precisions=[r["precision"] for r in sweep_results],
)

print("\nFINAL EVALUATION SUMMARY")
print(f"best_threshold={best_thr:.2f}")
print(f"precision={best_result['precision']:.4f}")
print(f"recall={best_result['recall']:.4f}")
print(f"f1={best_result['f1']:.4f}")
print(f"mean_iou={best_result['mean_iou']:.4f}")
print(f"approx_ap50={ap50:.4f}")
print(f"total_tp={best_result['total_tp']}")
print(f"total_fp={best_result['total_fp']}")
print(f"total_fn={best_result['total_fn']}")

In [ ]:
sweep_results = []
for thr in THRESHOLDS:
    res = evaluate_at_threshold(
        raw_predictions,
        threshold=thr,
        match_iou_threshold=EVAL_IOU_THRESHOLD,
    )
    sweep_results.append(res)

best_result = max(sweep_results, key=lambda x: x["f1"])
best_thr = best_result["threshold"]
ap50 = compute_ap_from_pr(
    recalls=[r["recall"] for r in sweep_results],
    precisions=[r["precision"] for r in sweep_results],
)

print("\nFINAL EVALUATION SUMMARY")
print(f"best_threshold={best_thr:.2f}")
print(f"precision={best_result['precision']:.4f}")
print(f"recall={best_result['recall']:.4f}")
print(f"f1={best_result['f1']:.4f}")
print(f"mean_iou={best_result['mean_iou']:.4f}")
print(f"approx_ap50={ap50:.4f}")
print(f"total_tp={best_result['total_tp']}")
print(f"total_fp={best_result['total_fp']}")
print(f"total_fn={best_result['total_fn']}")

In [ ]:
thresholds = [r["threshold"] for r in sweep_results]
precisions = [r["precision"] for r in sweep_results]
recalls = [r["recall"] for r in sweep_results]
f1s = [r["f1"] for r in sweep_results]
ious = [r["mean_iou"] for r in sweep_results]

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(recalls, precisions, marker="o")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve")
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(thresholds, precisions, marker="o", label="Precision")
plt.plot(thresholds, recalls, marker="o", label="Recall")
plt.xlabel("Confidence threshold")
plt.ylabel("Score")
plt.title("Precision and Recall vs confidence threshold")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(thresholds, ious, marker="o")
plt.xlabel("Confidence threshold")
plt.ylabel("Mean IoU")
plt.title("Mean IoU vs confidence threshold")
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
if len(best_result["tp_scores"]) > 0:
    plt.hist(best_result["tp_scores"], bins=15, alpha=0.7, label="TP scores")
if len(best_result["fp_scores"]) > 0:
    plt.hist(best_result["fp_scores"], bins=15, alpha=0.7, label="FP scores")
plt.xlabel("Confidence score")
plt.ylabel("Count")
plt.title(f"Confidence score distribution at threshold={best_thr:.2f}")
plt.legend()
plt.grid()
plt.show()

In [ ]:
epochs = []
train_loss = []
val_loss = []
learning_rate = []

with open(TRAINING_CSV_PATH, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for i, row in enumerate(reader):
        epochs.append(i)
        train_loss.append(float(row["loss"]))
        if "val_loss" in row and row["val_loss"] != "":
            val_loss.append(float(row["val_loss"]))
        if "learning_rate" in row and row["learning_rate"] != "":
            learning_rate.append(float(row["learning_rate"]))

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(epochs, train_loss, label="Training loss")
if len(val_loss) == len(epochs):
    plt.plot(epochs, val_loss, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
if len(learning_rate) == len(epochs):
    plt.figure(figsize=(7, 5))
    plt.plot(epochs, learning_rate)
    plt.xlabel("Epoch")
    plt.ylabel("Learning rate")
    plt.title("Learning rate schedule")
    plt.grid()
    plt.show()